In [2]:
import numpy as np 
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [17]:
df = pd.read_csv('covid_toy.csv')

In [18]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [19]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [20]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [21]:
X_train

,age,gender,fever,cough,city
78,11,Male,100.0,Mild,Bangalore
0,60,Male,103.0,Mild,Kolkata
11,65,Female,98.0,Mild,Mumbai
31,83,Male,103.0,Mild,Kolkata
75,5,Male,102.0,Mild,Kolkata
...,...,...,...,...,...
54,60,Female,99.0,Mild,Mumbai
98,5,Female,98.0,Strong,Mumbai
5,84,Female,NaN,Mild,Bangalore
95,12,Female,104.0,Mild,Bangalore


## 1. Aam Zindagi

In [22]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [24]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [25]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [26]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [27]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

## Mentos Zindagi

In [28]:
from sklearn.compose import ColumnTransformer

In [31]:
transformer =ColumnTransformer(transformers = [
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories = [['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output = False,drop='first',),['gender','city'])
],remainder='passthrough')

In [33]:
df_new = transformer.fit_transform(X_train)

In [37]:
df_new

array([[100.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  11.        ],
       [103.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  60.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  65.        ],
       [103.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  83.        ],
       [102.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   5.        ],
       [104.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  51.        ],
       [100.90277778,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  34.        ],
       [ 99.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  25.        ],
       [104.        ,   1.        ,   0.        ,   0.        ,
          0.    

In [38]:
df_new = pd.DataFrame(
    df_new, 
    columns=transformer.get_feature_names_out()
)

In [39]:
df_new.head()

,tnf1__fever,tnf2__cough,tnf3__gender_Male,tnf3__city_Delhi,tnf3__city_Kolkata,tnf3__city_Mumbai,remainder__age
0,100.0,0.0,1.0,0.0,0.0,0.0,11.0
1,103.0,0.0,1.0,0.0,1.0,0.0,60.0
2,98.0,0.0,0.0,0.0,0.0,1.0,65.0
3,103.0,0.0,1.0,0.0,1.0,0.0,83.0
4,102.0,0.0,1.0,0.0,1.0,0.0,5.0


In [21]:
df_new = pd.DataFrame(
    transformer.fit_transform(X_train),
    columns=transformer.get_feature_names_out()
)

In [24]:
df_new.head()

,tnf1__fever,tnf2__cough,tnf3__gender_Male,tnf3__city_Delhi,tnf3__city_Kolkata,tnf3__city_Mumbai,remainder__age
0,100.0,0.0,1.0,0.0,0.0,0.0,10.0
1,101.0,1.0,0.0,0.0,1.0,0.0,51.0
2,100.0,0.0,1.0,1.0,0.0,0.0,27.0
3,98.0,1.0,0.0,0.0,1.0,0.0,71.0
4,98.0,1.0,0.0,0.0,0.0,1.0,5.0


In [26]:
df_new = pd.DataFrame(df_new,columns = df_new.columns )
df_new

,tnf1__fever,tnf2__cough,tnf3__gender_Male,tnf3__city_Delhi,tnf3__city_Kolkata,tnf3__city_Mumbai,remainder__age
0,100.0,0.0,1.0,0.0,0.0,0.0,10.0
1,101.0,1.0,0.0,0.0,1.0,0.0,51.0
2,100.0,0.0,1.0,1.0,0.0,0.0,27.0
3,98.0,1.0,0.0,0.0,1.0,0.0,71.0
4,98.0,1.0,0.0,0.0,0.0,1.0,5.0
...,...,...,...,...,...,...,...
75,103.0,0.0,0.0,1.0,0.0,0.0,73.0
76,103.0,1.0,1.0,0.0,0.0,0.0,46.0
77,100.0,1.0,0.0,0.0,0.0,0.0,47.0
78,102.0,0.0,1.0,0.0,0.0,0.0,64.0


In [28]:
pd.get_dummies(df,columns=['cough','city'])

,age,gender,fever,has_covid,cough_Mild,cough_Strong,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai
0,60,Male,103.0,No,True,False,False,False,True,False
1,27,Male,100.0,Yes,True,False,False,True,False,False
2,42,Male,101.0,No,True,False,False,True,False,False
3,31,Female,98.0,No,True,False,False,False,True,False
4,65,Female,101.0,No,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...
95,12,Female,104.0,No,True,False,True,False,False,False
96,51,Female,101.0,Yes,False,True,False,False,True,False
97,20,Female,101.0,No,True,False,True,False,False,False
98,5,Female,98.0,No,False,True,False,False,False,True
